# Smart Inventory — Train / Validation / Test Split

Chronological splitting and categorical encoding for demand forecasting.

**Important:** Do not use random `train_test_split`. The model must learn from past data and be evaluated on future data.

**Target:** `Units Sold`  
**Excluded:** `Demand Forecast` (leakage), `Units Ordered` (initially excluded)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import joblib
from pathlib import Path

pd.set_option('display.max_columns', None)
print('Libraries imported successfully!')

Libraries imported successfully!


## 1. Load and feature-engineer the dataset

In [2]:
df = pd.read_csv("../DATA/retail_store_inventory.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)

group_cols = ["Store ID", "Product ID"]

# Calendar
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

# Lag features
df["Lag_1"] = df.groupby(group_cols)["Units Sold"].shift(1)
df["Lag_7"] = df.groupby(group_cols)["Units Sold"].shift(7)
df["Lag_14"] = df.groupby(group_cols)["Units Sold"].shift(14)
df["Lag_28"] = df.groupby(group_cols)["Units Sold"].shift(28)

# Rolling features
df["Rolling_Mean_7"] = df.groupby(group_cols)["Units Sold"].transform(lambda x: x.shift(1).rolling(7).mean())
df["Rolling_Mean_14"] = df.groupby(group_cols)["Units Sold"].transform(lambda x: x.shift(1).rolling(14).mean())
df["Rolling_Mean_28"] = df.groupby(group_cols)["Units Sold"].transform(lambda x: x.shift(1).rolling(28).mean())
df["Rolling_Std_7"] = df.groupby(group_cols)["Units Sold"].transform(lambda x: x.shift(1).rolling(7).std())

df["Price_Difference"] = df["Price"] - df["Competitor Pricing"]

time_series_features = [
    "Lag_1", "Lag_7", "Lag_14", "Lag_28",
    "Rolling_Mean_7", "Rolling_Mean_14",
    "Rolling_Mean_28", "Rolling_Std_7"
]

df_model = df.dropna(subset=time_series_features).copy()
df_model = df_model.sort_values("Date").reset_index(drop=True)

print("Model shape:", df_model.shape)
print("Start:", df_model["Date"].min())
print("End:", df_model["Date"].max())

Model shape: (70300, 29)
Start: 2022-01-29 00:00:00
End: 2024-01-01 00:00:00


## 2. Define X and y

In [3]:
feature_columns = [
    "Store ID", "Product ID", "Category", "Region",
    "Inventory Level", "Price", "Discount",
    "Weather Condition", "Holiday/Promotion",
    "Competitor Pricing", "Seasonality",
    "Year", "Month", "Day", "Day_of_Week", "Week_of_Year",
    "Lag_1", "Lag_7", "Lag_14", "Lag_28",
    "Rolling_Mean_7", "Rolling_Mean_14", "Rolling_Mean_28",
    "Rolling_Std_7", "Price_Difference"
]

target_column = "Units Sold"

X = df_model[feature_columns].copy()
y = df_model[target_column].copy()

print("Features:", len(feature_columns))
print("X:", X.shape)
print("y:", y.shape)

Features: 25
X: (70300, 25)
y: (70300,)


## 3. Calculate chronological split dates

We split by **unique dates**, so all Store/Product observations belonging to one date stay together.

In [4]:
unique_dates = np.sort(df_model["Date"].unique())
n_dates = len(unique_dates)

train_end_idx = int(n_dates * 0.70)
validation_end_idx = int(n_dates * 0.85)

train_end_date = unique_dates[train_end_idx - 1]
validation_end_date = unique_dates[validation_end_idx - 1]

print("Unique dates:", n_dates)
print("Train ends:", train_end_date)
print("Validation ends:", validation_end_date)
print("Test starts:", unique_dates[validation_end_idx])

Unique dates: 703
Train ends: 2023-06-04T00:00:00.000000
Validation ends: 2023-09-17T00:00:00.000000
Test starts: 2023-09-18T00:00:00.000000


## 4. Create train / validation / test sets

In [5]:
train_mask = df_model["Date"] <= train_end_date
val_mask = (df_model["Date"] > train_end_date) & (df_model["Date"] <= validation_end_date)
test_mask = df_model["Date"] > validation_end_date

X_train = df_model.loc[train_mask, feature_columns].copy()
y_train = df_model.loc[train_mask, target_column].copy()

X_val = df_model.loc[val_mask, feature_columns].copy()
y_val = df_model.loc[val_mask, target_column].copy()

X_test = df_model.loc[test_mask, feature_columns].copy()
y_test = df_model.loc[test_mask, target_column].copy()

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (49200, 25) (49200,)
Validation: (10500, 25) (10500,)
Test: (10600, 25) (10600,)


## 5. Verify chronological ordering

In [6]:
train_dates = df_model.loc[train_mask, "Date"]
val_dates = df_model.loc[val_mask, "Date"]
test_dates = df_model.loc[test_mask, "Date"]

print("TRAIN:", train_dates.min(), "→", train_dates.max())
print("VALIDATION:", val_dates.min(), "→", val_dates.max())
print("TEST:", test_dates.min(), "→", test_dates.max())

assert train_dates.max() < val_dates.min()
assert val_dates.max() < test_dates.min()

print("\nChronological split check passed!")

TRAIN: 2022-01-29 00:00:00 → 2023-06-04 00:00:00
VALIDATION: 2023-06-05 00:00:00 → 2023-09-17 00:00:00
TEST: 2023-09-18 00:00:00 → 2024-01-01 00:00:00

Chronological split check passed!


## 6. Identify categorical and numerical features

In [7]:
categorical_features = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Categorical:")
print(categorical_features)
print("\nNumerical:")
print(numerical_features)

Categorical:
['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']

Numerical:
['Inventory Level', 'Price', 'Discount', 'Holiday/Promotion', 'Competitor Pricing', 'Year', 'Month', 'Day', 'Day_of_Week', 'Week_of_Year', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_28', 'Rolling_Std_7', 'Price_Difference']


## 7. One-hot encode categorical features

The encoder is fitted **only on training data** to avoid leakage. `handle_unknown='ignore'` makes deployment safer if an unseen category appears.

In [8]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_val_cat = encoder.transform(X_val[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

print("Encoded categorical columns:", X_train_cat.shape[1])

Encoded categorical columns: 42


## 8. Combine numerical and categorical features

In [9]:
X_train_num = X_train[numerical_features].to_numpy()
X_val_num = X_val[numerical_features].to_numpy()
X_test_num = X_test[numerical_features].to_numpy()

X_train_encoded = np.hstack([X_train_num, X_train_cat])
X_val_encoded = np.hstack([X_val_num, X_val_cat])
X_test_encoded = np.hstack([X_test_num, X_test_cat])

print("X_train:", X_train_encoded.shape)
print("X_val:", X_val_encoded.shape)
print("X_test:", X_test_encoded.shape)

assert X_train_encoded.shape[1] == X_val_encoded.shape[1] == X_test_encoded.shape[1]
print("Feature dimension check passed!")

X_train: (49200, 61)
X_val: (10500, 61)
X_test: (10600, 61)
Feature dimension check passed!


## 9. Save prepared datasets and preprocessing

In [10]:
models_dir = Path("../Models")
models_dir.mkdir(parents=True, exist_ok=True)

np.save(models_dir / "X_train.npy", X_train_encoded)
np.save(models_dir / "X_val.npy", X_val_encoded)
np.save(models_dir / "X_test.npy", X_test_encoded)

np.save(models_dir / "y_train.npy", y_train.to_numpy())
np.save(models_dir / "y_val.npy", y_val.to_numpy())
np.save(models_dir / "y_test.npy", y_test.to_numpy())

joblib.dump(encoder, models_dir / "categorical_encoder.pkl")
joblib.dump({
    "feature_columns": feature_columns,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "target_column": target_column
}, models_dir / "feature_metadata.pkl")

print("Prepared datasets and encoder saved to ../Models/")

Prepared datasets and encoder saved to ../Models/


## Next step

The data is now ready for model training. Next we will build a baseline and train **XGBoost**, then evaluate with MAE, RMSE and R². The test set must remain untouched until final evaluation.